# Data Modeling Part I: Foundations and Intuition

## Objective
This notebook begins the modeling stage of the project. The goal is to use the final feature table to build a first predictive model that relates sanctions and contextual features to a civilian well-being outcome.

For this project, I use a regression approach with `child_mortality_u5` as the target variable. This target is meaningful because it captures an important aspect of civilian well-being and allows the model to study how sanctions, trade exposure, political conditions, and economic indicators relate to social outcomes.

This notebook focuses on:
- loading the final model-ready dataset
- defining the target and feature set
- implementing a proper train/validation/test split
- training a baseline model
- comparing training and validation performance
- interpreting whether the model is underfitting or overfitting

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load the Final Model-Ready Dataset

I begin by loading the final model-ready dataset created in the wrangling notebook. This file already contains the cleaned and engineered features needed for modeling.

At this stage, I check the dataset shape and preview the first few rows to confirm that the file loaded correctly and is ready for defining the target variable.

In [2]:
model_df = pd.read_csv("../data/processed/model_ready_dataset.csv")

print("Model-ready dataset shape:", model_df.shape)
model_df.head()

Model-ready dataset shape: (7820, 44)


,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,SH.DYN.MORT.MA,poverty_rate,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
0,1995,3.361391,2.547144,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
1,1996,3.225288,1.185789,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
2,1997,2.999948,7.046875,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
3,1998,1.869489,1.991984,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
4,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False


## Define the Modeling Target

For the first model, I use `child_mortality_u5` as the target variable. This is a meaningful regression target for the project because it represents an important dimension of civilian well-being.

Before training a model, I check how many rows have a valid target value. Rows with missing target values cannot be used for supervised learning, so they will be excluded from the modeling dataset.

In [3]:
target_col = "child_mortality_u5"

print("Missing target values:", model_df[target_col].isna().sum())
print("Non-missing target values:", model_df[target_col].notna().sum())

model_df = model_df[model_df[target_col].notna()].copy()

print("Shape after dropping missing target rows:", model_df.shape)

Missing target values: 502
Non-missing target values: 7318
Shape after dropping missing target rows: (7318, 44)


## Separate Features and Target

After selecting the target variable, I separate the dataset into:
- `X`: predictor features
- `y`: target values

I exclude `child_mortality_u5` from the feature matrix because it is the outcome the model is trying to predict.

In [4]:
X = model_df.drop(columns=[target_col])
y = model_df[target_col]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (7318, 43)
Target vector shape: (7318,)


## Handle Remaining Missing Values in Predictors

The target variable is now complete, but several predictor columns still contain missing values. Since baseline models like linear regression cannot train directly on missing data, I need a simple and reproducible imputation strategy.

For this first model, I use median imputation for numeric features. The median is a reasonable baseline choice because it is simple, robust to extreme values, and easy to explain.

In [5]:
# convert boolean columns to integers for modeling
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

# median imputation for numeric columns
X = X.apply(lambda col: col.fillna(col.median()) if pd.api.types.is_numeric_dtype(col) else col)

print("Remaining missing values in X:", X.isna().sum().sum())
X.head()

Remaining missing values in X: 0


,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,SH.DYN.MORT.FE,SH.DYN.MORT.MA,poverty_rate,gini_index,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
30,1995,12.288591,4.354618,92.03204,90.64573,91.5114,147.173735,164.943896,9.0,37.4,...,0.0,-0.019528,19.394568,18.949745,20.547127,21.517961,1,0,0,0
31,1996,9.706586,5.499111,92.03204,90.64573,91.5114,144.395247,162.164025,9.0,37.4,...,0.0,-0.092271,19.394568,18.949745,20.547127,21.517961,1,0,0,0
32,1997,10.249599,3.856739,92.03204,90.64573,91.5114,141.353445,159.039326,9.0,37.4,...,0.0,-0.039716,19.394568,18.949745,20.547127,21.517961,1,0,0,0
33,1998,7.495255,1.775172,92.03204,90.64573,91.5114,140.973185,158.406724,9.0,37.4,...,0.0,0.040731,19.394568,18.949745,20.547127,21.517961,1,0,0,0
34,1999,7.819865,2.634184,92.03204,90.64573,91.5114,132.727359,149.999585,9.0,37.4,...,0.0,0.047292,19.394568,18.949745,20.547127,21.517961,1,0,0,0


## Create a Proper Train / Validation / Test Split

To evaluate generalization properly, I split the data into three parts:
- training set
- validation set
- test set

Because this is a country-year dataset, I avoid a random split and instead use a time-based split by year. This is more defensible for the project because it evaluates whether the model trained on earlier years can generalize to later years.

I use:
- training: years before 2011
- validation: 2011 to 2014
- test: 2015 and later

In [6]:
train_df = model_df[model_df["year"] < 2011].copy()
val_df = model_df[(model_df["year"] >= 2011) & (model_df["year"] <= 2014)].copy()
test_df = model_df[model_df["year"] >= 2015].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain year range:", train_df["year"].min(), "to", train_df["year"].max())
print("Validation year range:", val_df["year"].min(), "to", val_df["year"].max())
print("Test year range:", test_df["year"].min(), "to", test_df["year"].max())

Train shape: (3904, 44)
Validation shape: (976, 44)
Test shape: (2438, 44)

Train year range: 1995 to 2010
Validation year range: 2011 to 2014
Test year range: 2015 to 2024


## Build Training, Validation, and Test Matrices

After splitting the dataset by year, I now create separate feature matrices and target vectors for each split.

This keeps the modeling workflow clear:
- `X_train`, `y_train`
- `X_val`, `y_val`
- `X_test`, `y_test`

I also apply the same simple median imputation to each split so that the baseline model can train without missing predictor values.

In [7]:
# separate X and y for each split
X_train = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].copy()

X_val = val_df.drop(columns=[target_col]).copy()
y_val = val_df[target_col].copy()

X_test = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].copy()

# convert boolean columns to integers
for df_split in [X_train, X_val, X_test]:
    bool_cols = df_split.select_dtypes(include=["bool"]).columns
    df_split[bool_cols] = df_split[bool_cols].astype(int)

# median imputation using training medians
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nRemaining missing values:")
print("X_train:", X_train.isna().sum().sum())
print("X_val:", X_val.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

X_train shape: (3904, 43)
y_train shape: (3904,)
X_val shape: (976, 43)
y_val shape: (976,)
X_test shape: (2438, 43)
y_test shape: (2438,)

Remaining missing values:
X_train: 0
X_val: 0
X_test: 0


## Train a Baseline Model

For the first model, I use **Linear Regression** as a baseline. This is a good starting point because it is simple, interpretable, and appropriate for a continuous target variable such as `child_mortality_u5`.

The goal of this first model is not necessarily to achieve the best possible performance. Instead, it provides a reference point for understanding whether the feature set contains useful signal and whether the model is underfitting or overfitting.

In [8]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

print("Baseline Linear Regression model trained successfully.")

Baseline Linear Regression model trained successfully.


## Generate Predictions and Evaluate Performance

After fitting the baseline model, I evaluate its performance on both the training set and the validation set. Comparing these two results helps diagnose whether the model is generalizing well or whether it is underfitting or overfitting.

For this regression problem, I report:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R²)

Lower MAE and RMSE are better, while higher R² is better.

In [9]:
# predictions
y_train_pred = baseline_model.predict(X_train)
y_val_pred = baseline_model.predict(X_val)

# metrics function
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

train_mae, train_rmse, train_r2 = regression_metrics(y_train, y_train_pred)
val_mae, val_rmse, val_r2 = regression_metrics(y_val, y_val_pred)

print("Training Performance:")
print(f"MAE:  {train_mae:.4f}")
print(f"RMSE: {train_rmse:.4f}")
print(f"R²:   {train_r2:.4f}")

print("\nValidation Performance:")
print(f"MAE:  {val_mae:.4f}")
print(f"RMSE: {val_rmse:.4f}")
print(f"R²:   {val_r2:.4f}")

Training Performance:
MAE:  0.0305
RMSE: 0.0393
R²:   1.0000

Validation Performance:
MAE:  0.0319
RMSE: 0.0413
R²:   1.0000


## Diagnose Possible Data Leakage

The baseline model produced nearly perfect training and validation performance, which is unrealistic for a real-world social outcome. This suggests that one or more predictor variables may contain direct information about the target.

Because the target is `child_mortality_u5`, closely related mortality variables such as `SH.DYN.MORT.FE` and `SH.DYN.MORT.MA` may be leaking target information into the model. To investigate this, I inspect which features are most strongly associated with the target.

In [10]:
corr_with_target = model_df.corr(numeric_only=True)[target_col].sort_values(ascending=False)

print("Top positive correlations with child_mortality_u5:")
print(corr_with_target.head(15))

print("\nTop negative correlations with child_mortality_u5:")
print(corr_with_target.tail(15))

Top positive correlations with child_mortality_u5:
child_mortality_u5          1.000000
SH.DYN.MORT.MA              0.999553
SH.DYN.MORT.FE              0.999404
child_mortality_u5_lag1     0.978573
child_mortality_gap         0.826218
poverty_rate                0.795178
poverty_rate_lag1           0.793685
gini_index                  0.313354
conflict_incidence          0.284029
conflict_intensity          0.279449
gdp_growth                  0.120224
gdp_growth_lag1             0.118633
inflation_rate_lag1         0.105903
inflation_rate              0.104643
unemployment_rate_change    0.045243
Name: child_mortality_u5, dtype: float64

Top negative correlations with child_mortality_u5:
unemployment_rate_lag1     -0.097199
oil_exports                -0.164431
fuel_imports               -0.177038
pharma_imports             -0.179505
total_trade_exposure       -0.214495
year                       -0.287695
log_fuel_imports           -0.324285
log_pharma_imports         -0.335616
log_o

## Remove Leakage-Prone Features

The correlation analysis shows that some predictors are too closely related to the target and are likely causing leakage or near-leakage. In particular, male and female child mortality variables are almost direct components of the target.

To make the model more defensible, I remove these leakage-prone features and retrain the baseline model using a cleaner feature set.

In [11]:
leakage_cols = [
    "SH.DYN.MORT.FE",
    "SH.DYN.MORT.MA",
    "child_mortality_u5_lag1",
    "child_mortality_gap"
]

X_train_clean = X_train.drop(columns=leakage_cols, errors="ignore")
X_val_clean = X_val.drop(columns=leakage_cols, errors="ignore")
X_test_clean = X_test.drop(columns=leakage_cols, errors="ignore")

print("Original X_train shape:", X_train.shape)
print("Cleaned X_train shape:", X_train_clean.shape)
print("Original X_val shape:", X_val.shape)
print("Cleaned X_val shape:", X_val_clean.shape)

Original X_train shape: (3904, 43)
Cleaned X_train shape: (3904, 39)
Original X_val shape: (976, 43)
Cleaned X_val shape: (976, 39)


## Retrain the Baseline Model Without Leakage Features

After removing the leakage-prone predictors, I retrain the linear regression model using the cleaned feature set. This provides a more honest estimate of how well the model can learn from sanctions, economic indicators, trade exposure, and political context without relying on features that are almost direct proxies for the target.

In [12]:
baseline_model_clean = LinearRegression()
baseline_model_clean.fit(X_train_clean, y_train)

print("Leakage-reduced baseline model trained successfully.")

Leakage-reduced baseline model trained successfully.


## Re-evaluate Training and Validation Performance

I now evaluate the leakage-reduced baseline model on the training and validation sets again. This comparison is more meaningful because the model no longer has access to features that are almost direct components of the target.

These results will help determine whether the earlier near-perfect performance was caused by leakage and whether the revised model shows a more realistic pattern of generalization.

In [13]:
# predictions from leakage-reduced model
y_train_pred_clean = baseline_model_clean.predict(X_train_clean)
y_val_pred_clean = baseline_model_clean.predict(X_val_clean)

# metrics
train_mae_clean, train_rmse_clean, train_r2_clean = regression_metrics(y_train, y_train_pred_clean)
val_mae_clean, val_rmse_clean, val_r2_clean = regression_metrics(y_val, y_val_pred_clean)

print("Training Performance (Leakage-Reduced Model):")
print(f"MAE:  {train_mae_clean:.4f}")
print(f"RMSE: {train_rmse_clean:.4f}")
print(f"R²:   {train_r2_clean:.4f}")

print("\nValidation Performance (Leakage-Reduced Model):")
print(f"MAE:  {val_mae_clean:.4f}")
print(f"RMSE: {val_rmse_clean:.4f}")
print(f"R²:   {val_r2_clean:.4f}")

Training Performance (Leakage-Reduced Model):
MAE:  17.6569
RMSE: 26.2193
R²:   0.7503

Validation Performance (Leakage-Reduced Model):
MAE:  14.1655
RMSE: 20.6068
R²:   0.6699


## Diagnose Underfitting vs Overfitting

The leakage-reduced results are more realistic than the earlier near-perfect model. To interpret them, I compare training and validation performance.

If both training and validation performance are poor, that suggests underfitting. If training performance is much better than validation performance, that suggests overfitting. If both are reasonably strong and fairly close, that suggests the model is learning useful structure without memorizing the training set.

In [14]:
performance_summary = pd.DataFrame({
    "Split": ["Train", "Validation"],
    "MAE": [train_mae_clean, val_mae_clean],
    "RMSE": [train_rmse_clean, val_rmse_clean],
    "R2": [train_r2_clean, val_r2_clean]
})

performance_summary

,Split,MAE,RMSE,R2
0,Train,17.656898,26.219257,0.750258
1,Validation,14.165539,20.606761,0.669897


## Interpretation of Model Behavior

The leakage-reduced linear regression model performs substantially worse than the earlier near-perfect model, which is a good sign because it suggests that the unrealistic performance was caused by leakage.

The revised model still performs reasonably well on validation data, which indicates that the engineered features contain useful signal for predicting child mortality. Training performance is somewhat better than validation performance, but the gap is not large enough to suggest severe overfitting. At the same time, the model is not so weak that it appears to be underfitting badly.

Overall, this baseline model appears to provide a reasonable starting point for the project: it is simple, interpretable, and able to generalize with moderate success.

In [15]:
coef_df = pd.DataFrame({
    "feature": X_train_clean.columns,
    "coefficient": baseline_model_clean.coef_
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("Top 15 features by absolute coefficient:")
coef_df[["feature", "coefficient"]].head(15)

Top 15 features by absolute coefficient:


,feature,coefficient
5,SE.PRM.NENR.MA,35.963356
4,SE.PRM.NENR.FE,-35.181145
18,school_enrollment_gap,33.799927
21,sanction_x_conflict,-26.463905
14,conflict_intensity,15.452010
20,sanction_x_regime,13.930924
12,regime_score,-11.134235
13,conflict_incidence,-4.764634
36,sanction_type_targeted,2.120758
3,school_enrollment,-2.047408


## Interpreting Coefficients Carefully

The coefficient table helps identify which predictors are most strongly associated with the target in the baseline linear regression model. However, these coefficients should be interpreted with caution.

First, coefficients in a linear model represent associations, not causal effects. A positive or negative coefficient does not prove that a feature directly causes child mortality to increase or decrease.

Second, some predictors in this dataset are conceptually related to each other, especially the education variables such as `SE.PRM.NENR.FE`, `SE.PRM.NENR.MA`, and `school_enrollment_gap`. When predictors are correlated, coefficient values can become unstable or harder to interpret individually. For that reason, the coefficient table is most useful as a rough indicator of influential variables rather than a causal ranking.

## Test Set Evaluation

After selecting the leakage-reduced baseline model based on training and validation results, I evaluate it once on the held-out test set.

This is the final generalization check. The test set represents later unseen years, so it provides a more honest estimate of how well the model may perform beyond the data used for training and validation.

In [16]:
# test predictions
y_test_pred_clean = baseline_model_clean.predict(X_test_clean)

# test metrics
test_mae_clean, test_rmse_clean, test_r2_clean = regression_metrics(y_test, y_test_pred_clean)

print("Test Performance (Leakage-Reduced Model):")
print(f"MAE:  {test_mae_clean:.4f}")
print(f"RMSE: {test_rmse_clean:.4f}")
print(f"R²:   {test_r2_clean:.4f}")

Test Performance (Leakage-Reduced Model):
MAE:  15.6048
RMSE: 21.0334
R²:   0.5245


## Compare Against a Simple Baseline

To make the model evaluation more meaningful, I compare the leakage-reduced linear regression model against a very simple baseline model. I use a `DummyRegressor`, which predicts the mean target value from the training set for every observation.

This provides a sanity check: if the trained model does not outperform this naive baseline, then the feature set or modeling approach is not adding useful predictive value.

In [17]:
from sklearn.dummy import DummyRegressor

# simple regression baseline
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train_clean, y_train)

# predictions from the dummy model
y_val_dummy = dummy_model.predict(X_val_clean)
y_test_dummy = dummy_model.predict(X_test_clean)

# metrics for comparison
dummy_val_mae, dummy_val_rmse, dummy_val_r2 = regression_metrics(y_val, y_val_dummy)
dummy_test_mae, dummy_test_rmse, dummy_test_r2 = regression_metrics(y_test, y_test_dummy)

baseline_comparison_df = pd.DataFrame({
    "Model": ["DummyRegressor", "Leakage-Reduced Linear Regression"],
    "Validation MAE": [dummy_val_mae, val_mae_clean],
    "Validation RMSE": [dummy_val_rmse, val_rmse_clean],
    "Validation R2": [dummy_val_r2, val_r2_clean],
    "Test MAE": [dummy_test_mae, test_mae_clean],
    "Test RMSE": [dummy_test_rmse, test_rmse_clean],
    "Test R2": [dummy_test_r2, test_r2_clean]
})

baseline_comparison_df

,Model,Validation MAE,Validation RMSE,Validation R2,Test MAE,Test RMSE,Test R2
0,DummyRegressor,34.802488,40.022906,-0.245225,34.300374,38.898642,-0.626363
1,Leakage-Reduced Linear Regression,14.165539,20.606761,0.669897,15.604810,21.033446,0.524479


The comparison above shows whether the leakage-reduced linear regression model improves on a trivial baseline that always predicts the training-set mean. This is an important benchmark because it confirms whether the engineered features and modeling workflow provide value beyond a naive prediction rule.

## Final Modeling Summary

The first baseline model began with unrealistically perfect performance, which indicated leakage from features that were too closely related to the target. After removing those leakage-prone variables, the linear regression model produced much more realistic results.

The leakage-reduced model achieved reasonably strong training performance and moderate validation and test performance. This suggests that the engineered features contain useful signal for predicting child mortality, while also showing that the problem remains challenging on unseen years.

In addition, the leakage-reduced linear regression model outperformed a simple `DummyRegressor` baseline, which indicates that the engineered predictors contribute meaningful predictive value beyond a naive mean-based prediction.

Overall, the model does not appear to be severely underfit or severely overfit. Instead, it provides a reasonable and interpretable baseline for later improvement.

## Save Predictions and Modeling Outputs

To support later visualization, I save the predictions and key performance results. These outputs can be used in the dashboard notebook for error analysis, model comparison, and communication of findings.

In [18]:
# save predictions
train_results = pd.DataFrame({
    "year": train_df["year"].values,
    "actual_child_mortality_u5": y_train.values,
    "predicted_child_mortality_u5": y_train_pred_clean
})

val_results = pd.DataFrame({
    "year": val_df["year"].values,
    "actual_child_mortality_u5": y_val.values,
    "predicted_child_mortality_u5": y_val_pred_clean
})

test_results = pd.DataFrame({
    "year": test_df["year"].values,
    "actual_child_mortality_u5": y_test.values,
    "predicted_child_mortality_u5": y_test_pred_clean
})

metrics_df = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "MAE": [train_mae_clean, val_mae_clean, test_mae_clean],
    "RMSE": [train_rmse_clean, val_rmse_clean, test_rmse_clean],
    "R2": [train_r2_clean, val_r2_clean, test_r2_clean]
})

coef_df.to_csv("../data/processed/linear_model_coefficients.csv", index=False)
train_results.to_csv("../data/processed/train_predictions.csv", index=False)
val_results.to_csv("../data/processed/validation_predictions.csv", index=False)
test_results.to_csv("../data/processed/test_predictions.csv", index=False)
metrics_df.to_csv("../data/processed/model_metrics.csv", index=False)

print("Saved: ../data/processed/linear_model_coefficients.csv")
print("Saved: ../data/processed/train_predictions.csv")
print("Saved: ../data/processed/validation_predictions.csv")
print("Saved: ../data/processed/test_predictions.csv")
print("Saved: ../data/processed/model_metrics.csv")

Saved: ../data/processed/linear_model_coefficients.csv
Saved: ../data/processed/train_predictions.csv
Saved: ../data/processed/validation_predictions.csv
Saved: ../data/processed/test_predictions.csv
Saved: ../data/processed/model_metrics.csv


## Reload Check for Saved Modeling Outputs

As a final verification step, I reload the saved metrics and test predictions to confirm that the modeling outputs were written correctly and are ready to be used in the visualization notebook.

In [19]:
saved_metrics = pd.read_csv("../data/processed/model_metrics.csv")
saved_test_preds = pd.read_csv("../data/processed/test_predictions.csv")

print("Saved metrics:")
display(saved_metrics)

print("\nSaved test predictions preview:")
saved_test_preds.head()

Saved metrics:


,split,MAE,RMSE,R2
0,train,17.656898,26.219257,0.750258
1,validation,14.165539,20.606761,0.669897
2,test,15.604810,21.033446,0.524479



Saved test predictions preview:


,year,actual_child_mortality_u5,predicted_child_mortality_u5
0,2015,71.824438,54.190383
1,2016,69.672206,52.989966
2,2017,68.239022,52.345967
3,2018,63.357697,51.845564
4,2019,60.658078,51.040441


## Conclusion

This notebook built a baseline regression model to predict `child_mortality_u5` using the final model-ready feature table. After identifying and removing leakage-prone predictors, the revised linear regression model produced more realistic training, validation, and test performance.

The model provides a reasonable and interpretable baseline for the project. Its saved predictions, metrics, and coefficients will be used in the dashboard notebook to communicate findings and analyze model behavior.

## Model Comparison and Evaluation Extension

To strengthen the modeling stage, I extend the evaluation by training an additional candidate model, comparing it with the leakage-reduced linear regression baseline, and using cross-validation for a more reliable estimate of performance.

This section focuses on:
- training a second regression model
- comparing model performance against the baseline
- using cross-validation
- understanding which features are most influential

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import permutation_importance

## Train a Second Candidate Model

To compare with the leakage-reduced linear regression model, I train a second candidate model: a Random Forest Regressor.

This model is useful because it can capture nonlinear relationships and interactions more flexibly than linear regression. By comparing the two models, I can see whether a more flexible model improves predictive performance on this problem.

In [21]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_clean, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


## Rebuild Modeling Variables

This cell rebuilds the cleaned train, validation, and test matrices so that the model comparison section can run even if the notebook kernel was restarted.

In [22]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# reload model-ready data
model_df = pd.read_csv("../data/processed/model_ready_dataset.csv")

# target
target_col = "child_mortality_u5"

# keep only rows with observed target
model_df = model_df[model_df[target_col].notna()].copy()

# time-based split
train_df = model_df[model_df["year"] < 2011].copy()
val_df = model_df[(model_df["year"] >= 2011) & (model_df["year"] <= 2014)].copy()
test_df = model_df[model_df["year"] >= 2015].copy()

# split X and y
X_train = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].copy()

X_val = val_df.drop(columns=[target_col]).copy()
y_val = val_df[target_col].copy()

X_test = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].copy()

# convert bool columns to int
for df_split in [X_train, X_val, X_test]:
    bool_cols = df_split.select_dtypes(include=["bool"]).columns
    df_split[bool_cols] = df_split[bool_cols].astype(int)

# median imputation using training medians
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# remove leakage-prone columns
leakage_cols = [
    "SH.DYN.MORT.FE",
    "SH.DYN.MORT.MA",
    "child_mortality_u5_lag1",
    "child_mortality_gap"
]

X_train_clean = X_train.drop(columns=leakage_cols, errors="ignore")
X_val_clean = X_val.drop(columns=leakage_cols, errors="ignore")
X_test_clean = X_test.drop(columns=leakage_cols, errors="ignore")

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

print("Recovered X_train_clean shape:", X_train_clean.shape)
print("Recovered X_val_clean shape:", X_val_clean.shape)
print("Recovered X_test_clean shape:", X_test_clean.shape)

Recovered X_train_clean shape: (3904, 39)
Recovered X_val_clean shape: (976, 39)
Recovered X_test_clean shape: (2438, 39)


## Evaluate the Second Candidate Model

After training the Random Forest model, I evaluate it on the training, validation, and test sets using the same regression metrics as before: MAE, RMSE, and R².

This makes the comparison with the leakage-reduced linear regression model fair and consistent.

## Rebuild Linear Regression Metrics for Comparison

This cell rebuilds the leakage-reduced linear regression model and its validation/test metrics so that both candidate models can be compared in the same notebook session.

In [23]:
# Random Forest predictions
y_train_pred_rf = rf_model.predict(X_train_clean)
y_val_pred_rf = rf_model.predict(X_val_clean)
y_test_pred_rf = rf_model.predict(X_test_clean)

# Random Forest metrics
train_mae_rf, train_rmse_rf, train_r2_rf = regression_metrics(y_train, y_train_pred_rf)
val_mae_rf, val_rmse_rf, val_r2_rf = regression_metrics(y_val, y_val_pred_rf)
test_mae_rf, test_rmse_rf, test_r2_rf = regression_metrics(y_test, y_test_pred_rf)

print("Random Forest Training Performance:")
print(f"MAE:  {train_mae_rf:.4f}")
print(f"RMSE: {train_rmse_rf:.4f}")
print(f"R²:   {train_r2_rf:.4f}")

print("\nRandom Forest Validation Performance:")
print(f"MAE:  {val_mae_rf:.4f}")
print(f"RMSE: {val_rmse_rf:.4f}")
print(f"R²:   {val_r2_rf:.4f}")

print("\nRandom Forest Test Performance:")
print(f"MAE:  {test_mae_rf:.4f}")
print(f"RMSE: {test_rmse_rf:.4f}")
print(f"R²:   {test_r2_rf:.4f}")

Random Forest Training Performance:
MAE:  4.3151
RMSE: 7.4016
R²:   0.9801

Random Forest Validation Performance:
MAE:  9.3638
RMSE: 15.6861
R²:   0.8087

Random Forest Test Performance:
MAE:  13.1282
RMSE: 20.6914
R²:   0.5398


In [24]:
from sklearn.linear_model import LinearRegression

# rebuild leakage-reduced linear regression
baseline_model_clean = LinearRegression()
baseline_model_clean.fit(X_train_clean, y_train)

# predictions
y_train_pred_clean = baseline_model_clean.predict(X_train_clean)
y_val_pred_clean = baseline_model_clean.predict(X_val_clean)
y_test_pred_clean = baseline_model_clean.predict(X_test_clean)

# metrics
train_mae_clean, train_rmse_clean, train_r2_clean = regression_metrics(y_train, y_train_pred_clean)
val_mae_clean, val_rmse_clean, val_r2_clean = regression_metrics(y_val, y_val_pred_clean)
test_mae_clean, test_rmse_clean, test_r2_clean = regression_metrics(y_test, y_test_pred_clean)

print("Leakage-reduced linear regression metrics rebuilt successfully.")
print("Validation R²:", round(val_r2_clean, 4))
print("Test R²:", round(test_r2_clean, 4))

Leakage-reduced linear regression metrics rebuilt successfully.
Validation R²: 0.6699
Test R²: 0.5245


## Compare the Two Candidate Models

I now compare the leakage-reduced linear regression model and the Random Forest model using the same validation and test metrics. This makes it easier to see which model performs better overall and whether the more flexible model provides a meaningful improvement.

In [25]:
model_comparison_df = pd.DataFrame({
    "Model": ["Leakage-Reduced Linear Regression", "Random Forest Regressor"],
    "Validation MAE": [val_mae_clean, val_mae_rf],
    "Validation RMSE": [val_rmse_clean, val_rmse_rf],
    "Validation R2": [val_r2_clean, val_r2_rf],
    "Test MAE": [test_mae_clean, test_mae_rf],
    "Test RMSE": [test_rmse_clean, test_rmse_rf],
    "Test R2": [test_r2_clean, test_r2_rf]
})

model_comparison_df

,Model,Validation MAE,Validation RMSE,Validation R2,Test MAE,Test RMSE,Test R2
0,Leakage-Reduced Linear Regression,14.165539,20.606761,0.669897,15.60481,21.033446,0.524479
1,Random Forest Regressor,9.363771,15.686107,0.808724,13.12824,20.691442,0.539818


## Cross-Validation for More Reliable Performance Estimates

To make the evaluation more robust, I use cross-validation on the training data. Cross-validation provides multiple performance estimates across different folds, which reduces the chance of relying too heavily on a single split.

Here I use 5-fold cross-validation and compare the leakage-reduced linear regression model with the Random Forest model using negative RMSE as the scoring metric.

In [26]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Linear Regression CV
lr_cv_rmse = -cross_val_score(
    baseline_model_clean,
    X_train_clean,
    y_train,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

# Random Forest CV
rf_cv_rmse = -cross_val_score(
    rf_model,
    X_train_clean,
    y_train,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

cv_results_df = pd.DataFrame({
    "Model": ["Leakage-Reduced Linear Regression", "Random Forest Regressor"],
    "CV Mean RMSE": [lr_cv_rmse.mean(), rf_cv_rmse.mean()],
    "CV Std RMSE": [lr_cv_rmse.std(), rf_cv_rmse.std()]
})

cv_results_df

,Model,CV Mean RMSE,CV Std RMSE
0,Leakage-Reduced Linear Regression,26.709743,1.099929
1,Random Forest Regressor,13.796635,2.256895


## Permutation Importance for Model Understanding

To better understand what the stronger model is relying on, I use permutation importance on the validation set for the Random Forest model.

Permutation importance measures how much model performance drops when a feature is randomly shuffled. If shuffling a feature causes a large drop in performance, that feature is likely important for the model.

In [27]:
perm_result = permutation_importance(
    rf_model,
    X_val_clean,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": X_val_clean.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values("importance_mean", ascending=False)

print("Top 15 features by permutation importance:")
perm_importance_df.head(15)

Top 15 features by permutation importance:


,feature,importance_mean,importance_std
4,SE.PRM.NENR.FE,15.001633,0.441776
6,poverty_rate,14.354935,1.032130
3,school_enrollment,4.671945,0.823890
18,school_enrollment_gap,2.910636,0.444790
8,unemployment_rate,1.355905,0.310381
12,regime_score,1.074551,0.206241
31,log_oil_exports,0.416412,0.142712
7,gini_index,0.384281,0.075195
5,SE.PRM.NENR.MA,0.362845,0.054466
17,fuel_imports,0.317197,0.196975


## Interpret the Model Comparison

The model comparison shows that the Random Forest Regressor performs better than the leakage-reduced linear regression model on validation, test, and cross-validation performance. This suggests that the relationship between the predictors and child mortality is not purely linear, and that a more flexible model can capture additional structure in the data.

At the same time, the improvement on the final test set is only modest, which suggests that the more complex model does not produce a dramatic increase in generalization. This means the Random Forest is the stronger model overall, but the linear regression model still remains useful as a simpler and more interpretable baseline.

In [28]:
full_comparison_df = model_comparison_df.merge(cv_results_df, on="Model", how="left")
full_comparison_df

,Model,Validation MAE,Validation RMSE,Validation R2,Test MAE,Test RMSE,Test R2,CV Mean RMSE,CV Std RMSE
0,Leakage-Reduced Linear Regression,14.165539,20.606761,0.669897,15.60481,21.033446,0.524479,26.709743,1.099929
1,Random Forest Regressor,9.363771,15.686107,0.808724,13.12824,20.691442,0.539818,13.796635,2.256895


## Save Extended Evaluation Outputs

To support later reporting and visualization, I save the extended model comparison results and the permutation importance table. These outputs make it easier to compare models and explain which features are most influential in the stronger model.

In [29]:
full_comparison_df.to_csv("../data/processed/extended_model_comparison.csv", index=False)
perm_importance_df.to_csv("../data/processed/random_forest_permutation_importance.csv", index=False)

print("Saved: ../data/processed/extended_model_comparison.csv")
print("Saved: ../data/processed/random_forest_permutation_importance.csv")

Saved: ../data/processed/extended_model_comparison.csv
Saved: ../data/processed/random_forest_permutation_importance.csv


## Evaluation Summary

This extended evaluation strengthens the modeling stage in several ways.

First, the leakage-reduced linear regression model was compared against a simple DummyRegressor baseline, confirming that the engineered feature set provides meaningful predictive value beyond a naive mean-based prediction.

Second, a Random Forest Regressor was trained as a second candidate model. The Random Forest achieved better validation, test, and cross-validation performance than the leakage-reduced linear regression model, indicating that a more flexible nonlinear model can capture additional structure in the data.

Third, permutation importance was used to interpret which variables the Random Forest model relies on most strongly. The most important features included education-related variables, poverty, unemployment, and political context, which are substantively meaningful for the project.

Overall, this stage shows that the project now includes:
- a simple baseline model
- a stronger second candidate model
- appropriate regression metrics
- baseline comparison
- cross-validation
- feature importance analysis

This makes the modeling notebook more complete and better aligned with evaluation and model understanding goals.